# Phase 1: Parse Log voi Drain3

Dataset: Loghub BGL (`BGL_2k.log`). Notebook nay thuc hien:

- Load log file va dem tong so dong.
- Parse toan bo log voi Drain3.
- Liet ke tat ca templates va so dong moi template.
- Export top-10 templates ra `results/top_templates.csv`.
- Tune `drain_sim_th` voi `0.3`, `0.5`, `0.7` va chon gia tri tot nhat.


In [4]:
from pathlib import Path
import pandas as pd
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

DATA_PATH = Path("logpai loghub master BGL/BGL_2k.log")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

assert DATA_PATH.exists(), f"Khong tim thay file log: {DATA_PATH}"


## 1. Load log file va dem tong so dong

In [5]:
log_lines = DATA_PATH.read_text(encoding="utf-8", errors="ignore").splitlines()
total_lines = len(log_lines)

print(f"Log file: {DATA_PATH}")
print(f"Tong so dong log: {total_lines:,}")
print("\n5 dong dau:")
for line in log_lines[:5]:
    print(line)


Log file: logpai loghub master BGL/BGL_2k.log
Tong so dong log: 2,000

5 dong dau:
- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.675872 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
- 1117838573 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.53.276129 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
- 1117838976 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.49.36.156884 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
- 1117838978 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.49.38.026704 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
- 1117842440 2005.06.03 R23-M0-NE-C:J05-U01 2005-06-03-16.47.20.730545 R23-M0-NE-C:J05-U01 RAS KERNEL INFO 63543 double-hummer alignment exceptions


## 2. Parse toan bo log voi Drain3

BGL co nhieu truong dong trong header nhu timestamp, node id, thoi gian, vi tri may. O day parse toan bo dong raw log de dung yeu cau assignment; Drain3 se bien cac phan thay doi thanh wildcard `<*>`.

In [6]:
def build_miner(sim_th=0.5, depth=4):
    config = TemplateMinerConfig()
    config.drain_sim_th = sim_th
    config.drain_depth = depth
    return TemplateMiner(config=config)


def parse_logs(lines, sim_th=0.5, depth=4):
    miner = build_miner(sim_th=sim_th, depth=depth)
    parsed_rows = []

    for line in lines:
        result = miner.add_log_message(line)
        parsed_rows.append({
            "template_id": result["cluster_id"],
            "template_mined_at_parse_time": result["template_mined"],
            "change_type": result["change_type"],
            "log_line": line,
        })

    parsed_df = pd.DataFrame(parsed_rows)

    # Drain3 co the generalize template cua cung cluster theo thoi gian.
    # Vi vay template_df dung final template trong miner.drain.clusters.
    template_df = pd.DataFrame([
        {
            "template_id": cluster.cluster_id,
            "template": cluster.get_template(),
            "count": cluster.size,
        }
        for cluster in miner.drain.clusters
    ])

    template_df = (
        template_df
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    final_template_lookup = dict(zip(template_df["template_id"], template_df["template"]))
    parsed_df["template"] = parsed_df["template_id"].map(final_template_lookup)

    return miner, parsed_df, template_df


DEFAULT_SIM_TH = 0.5
miner, parsed_df, template_df = parse_logs(log_lines, sim_th=DEFAULT_SIM_TH, depth=4)

print(f"Tong so dong da parse: {len(parsed_df):,}")
print(f"So final template unique voi drain_sim_th={DEFAULT_SIM_TH}: {len(template_df):,}")
template_df.head(10)


Tong so dong da parse: 2,000
So final template unique voi drain_sim_th=0.5: 151


,template_id,template,count
0,73,- <*> 2005.07.09 <*> <*> <*> RAS KERNEL INFO g...,180
1,85,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> floa...,121
2,2,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> doub...,109
3,3,- <*> <*> <*> <*> <*> RAS KERNEL INFO CE sym <...,92
4,77,- <*> 2005.07.13 <*> <*> <*> RAS KERNEL INFO g...,87
5,138,- <*> 2005.12.01 <*> <*> <*> RAS KERNEL INFO <...,71
6,119,- <*> 2005.11.04 <*> <*> <*> RAS KERNEL INFO i...,61
7,14,KERNDTLB <*> 2005.06.11 R30-M0-N9-C:J16-U01 <*...,60
8,118,- <*> 2005.11.03 <*> <*> <*> RAS KERNEL INFO i...,59
9,137,- <*> 2005.12.01 <*> <*> <*> RAS KERNEL INFO 0...,51


## 3. Liet ke tat ca templates va dem so dong moi template

In [7]:
template_df

,template_id,template,count
0,73,- <*> 2005.07.09 <*> <*> <*> RAS KERNEL INFO g...,180
1,85,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> floa...,121
2,2,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> doub...,109
3,3,- <*> <*> <*> <*> <*> RAS KERNEL INFO CE sym <...,92
4,77,- <*> 2005.07.13 <*> <*> <*> RAS KERNEL INFO g...,87
...,...,...,...
146,110,- 1126211906 2005.09.08 R62-M1-NC-I:J18-U11 20...,1
147,112,- 1127264516 2005.09.20 R05-M0-N1-C:J06-U11 20...,1
148,15,- 1118699850 2005.06.13 R22-M0-NC-I:J18-U01 20...,1
149,116,- 1129734520 2005.10.19 R17-M0-N0-I:J18-U01 20...,1


## 4. Export top-10 templates ra CSV

File output can co columns: `template_id`, `template`, `count`.

In [8]:
top_templates = template_df[["template_id", "template", "count"]].head(10)
top_templates_path = RESULTS_DIR / "top_templates.csv"
top_templates.to_csv(top_templates_path, index=False)

print(f"Da export: {top_templates_path}")
top_templates


Da export: results/top_templates.csv


,template_id,template,count
0,73,- <*> 2005.07.09 <*> <*> <*> RAS KERNEL INFO g...,180
1,85,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> floa...,121
2,2,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> doub...,109
3,3,- <*> <*> <*> <*> <*> RAS KERNEL INFO CE sym <...,92
4,77,- <*> 2005.07.13 <*> <*> <*> RAS KERNEL INFO g...,87
5,138,- <*> 2005.12.01 <*> <*> <*> RAS KERNEL INFO <...,71
6,119,- <*> 2005.11.04 <*> <*> <*> RAS KERNEL INFO i...,61
7,14,KERNDTLB <*> 2005.06.11 R30-M0-N9-C:J16-U01 <*...,60
8,118,- <*> 2005.11.03 <*> <*> <*> RAS KERNEL INFO i...,59
9,137,- <*> 2005.12.01 <*> <*> <*> RAS KERNEL INFO 0...,51


## 5. Tune `drain_sim_th`: thu 0.3, 0.5, 0.7

Tieu chi chon trong assignment nay:

- Khong qua it template: tranh gop nhieu event khac nhau vao cung mot template.
- Khong qua nhieu template: tranh moi bien the nho thanh template rieng.
- Top templates con doc duoc va co y nghia.

Voi dataset nho `BGL_2k`, gia tri can bang thuong la `0.5` neu so template khong tang qua manh so voi `0.3` va khong bi tach qua nhieu nhu `0.7`.

In [9]:
tuning_rows = []
tuning_templates = {}

for sim_th in [0.3, 0.5, 0.7]:
    _, _, tuned_template_df = parse_logs(log_lines, sim_th=sim_th, depth=4)
    tuning_templates[sim_th] = tuned_template_df
    tuning_rows.append({
        "drain_sim_th": sim_th,
        "num_templates": len(tuned_template_df),
        "top_template_count": int(tuned_template_df.iloc[0]["count"]),
        "top_template": tuned_template_df.iloc[0]["template"],
    })

tuning_df = pd.DataFrame(tuning_rows)
tuning_path = RESULTS_DIR / "drain_sim_th_tuning.csv"
tuning_df.to_csv(tuning_path, index=False)

print(f"Da export tuning summary: {tuning_path}")
tuning_df


Da export tuning summary: results/drain_sim_th_tuning.csv


,drain_sim_th,num_templates,top_template_count,top_template
0,0.3,73,729,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> <*>
1,0.5,151,180,- <*> 2005.07.09 <*> <*> <*> RAS KERNEL INFO g...
2,0.7,1459,71,- <*> 2005.12.01 <*> <*> <*> RAS KERNEL INFO <...


In [10]:
# Chon gia tri tot nhat.
# Cach chon tu dong o day uu tien 0.5 vi day la muc can bang pho bien cua Drain3.
# Neu 0.5 tao so template bat thuong qua cao/thap, hay xem bang tuning_df de dieu chinh lai.
best_sim_th = 0.5

print(f"Gia tri drain_sim_th duoc chon: {best_sim_th}")
print("Ly do: 0.5 la nguong can bang, thuong tranh duoc ca viec gop qua nhieu nhu 0.3 va tach qua nhieu nhu 0.7.")


Gia tri drain_sim_th duoc chon: 0.5
Ly do: 0.5 la nguong can bang, thuong tranh duoc ca viec gop qua nhieu nhu 0.3 va tach qua nhieu nhu 0.7.


# Phase 2: Anomaly Detection tren Log

Phase nay dung output cua Phase 1 (`parsed_df`, `template_df`) de:

- Extract timestamp va label tu BGL raw log.
- Tao template count time series voi window 5 phut.
- Apply 3-sigma detector tren count cua tung template.
- Detect template spike bat thuong.
- Liet ke template moi xuat hien lan dau khi nao.
- Neu dataset co label, tinh precision/recall muc log-line dua tren BGL label.


In [9]:
def parse_bgl_metadata(line):
    """Extract metadata tu mot dong BGL raw log.

    Format chinh cua BGL_2k:
    label epoch date node timestamp node RAS component severity message
    """
    parts = line.split(maxsplit=9)
    if len(parts) < 10:
        return {
            "label": None,
            "timestamp": pd.NaT,
            "severity": None,
            "message": line,
        }

    label, epoch_seconds, date, node_1, raw_ts, node_2, ras, component, severity, message = parts
    return {
        "label": label,
        "timestamp": pd.to_datetime(int(epoch_seconds), unit="s"),
        "raw_timestamp": raw_ts,
        "node": node_1,
        "component": component,
        "severity": severity,
        "message": message,
    }


metadata_df = pd.DataFrame([parse_bgl_metadata(line) for line in log_lines])
log_df = pd.concat([metadata_df, parsed_df[["template_id", "template"]]], axis=1)
log_df["is_labeled_anomaly"] = log_df["label"].ne("-")

print(f"Tong so dong: {len(log_df):,}")
print(f"So dong co label anomaly theo BGL: {log_df['is_labeled_anomaly'].sum():,}")
print(f"Time range: {log_df['timestamp'].min()} -> {log_df['timestamp'].max()}")
log_df.head()


Tong so dong: 2,000
So dong co label anomaly theo BGL: 143
Time range: 2005-06-03 22:42:50 -> 2006-01-03 15:13:09


,label,timestamp,raw_timestamp,node,component,severity,message,template_id,template,is_labeled_anomaly
0,-,2005-06-03 22:42:50,2005-06-03-15.42.50.675872,R02-M1-N0-C:J12-U11,KERNEL,INFO,instruction cache parity error corrected,1,- <*> <*> <*> <*> <*> RAS KERNEL INFO instruct...,False
1,-,2005-06-03 22:42:53,2005-06-03-15.42.53.276129,R02-M1-N0-C:J12-U11,KERNEL,INFO,instruction cache parity error corrected,1,- <*> <*> <*> <*> <*> RAS KERNEL INFO instruct...,False
2,-,2005-06-03 22:49:36,2005-06-03-15.49.36.156884,R02-M1-N0-C:J12-U11,KERNEL,INFO,instruction cache parity error corrected,1,- <*> <*> <*> <*> <*> RAS KERNEL INFO instruct...,False
3,-,2005-06-03 22:49:38,2005-06-03-15.49.38.026704,R02-M1-N0-C:J12-U11,KERNEL,INFO,instruction cache parity error corrected,1,- <*> <*> <*> <*> <*> RAS KERNEL INFO instruct...,False
4,-,2005-06-03 23:47:20,2005-06-03-16.47.20.730545,R23-M0-NE-C:J05-U01,KERNEL,INFO,63543 double-hummer alignment exceptions,2,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> doub...,False


## 6. Tao template count time series, window 5 phut

In [10]:
WINDOW = "5min"

template_ts = (
    log_df
    .groupby([pd.Grouper(key="timestamp", freq=WINDOW), "template_id"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

print(f"Time series shape: {template_ts.shape[0]:,} windows x {template_ts.shape[1]:,} templates")
template_ts.head()


Time series shape: 831 windows x 151 templates


template_id,1,2,3,4,5,6,7,8,9,10,...,142,143,144,145,146,147,148,149,150,151
timestamp,,,,,,,,,,,,,,,,,,,,,
2005-06-03 22:40:00,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2005-06-03 22:45:00,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2005-06-03 23:45:00,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2005-06-03 23:55:00,0,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2005-06-04 01:20:00,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 7. Apply 3-sigma anomaly detector tren template count

Voi moi template, tinh mean va std tren count theo window. Mot spike duoc flag neu:

```text
count > mean + 3 * std
```

In [11]:
def detect_template_spikes_3sigma(template_ts, template_lookup, sigma=3):
    rows = []

    for template_id in template_ts.columns:
        counts = template_ts[template_id].astype(float)
        mean = counts.mean()
        std = counts.std(ddof=0)

        if std == 0 or pd.isna(std):
            continue

        threshold = mean + sigma * std
        spike_points = counts[(counts > threshold) & (counts > 0)]

        for ts, count in spike_points.items():
            rows.append({
                "timestamp": ts,
                "template_id": template_id,
                "template": template_lookup.get(template_id, ""),
                "count": int(count),
                "mean": round(float(mean), 3),
                "std": round(float(std), 3),
                "threshold": round(float(threshold), 3),
                "z_score": round(float((count - mean) / std), 3),
            })

    return (
        pd.DataFrame(rows)
        .sort_values(["z_score", "count"], ascending=False)
        .reset_index(drop=True)
    )


template_lookup = dict(zip(template_df["template_id"], template_df["template"]))
spike_df = detect_template_spikes_3sigma(template_ts, template_lookup, sigma=3)
spike_path = RESULTS_DIR / "template_count_spikes_3sigma.csv"
spike_df.to_csv(spike_path, index=False)

print(f"So spike duoc detect: {len(spike_df):,}")
print(f"Da export: {spike_path}")
spike_df.head(20)


So spike duoc detect: 687
Da export: results/template_count_spikes_3sigma.csv


,timestamp,template_id,template,count,mean,std,threshold,z_score
0,2005-07-03 00:45:00,66,- <*> 2005.07.02 <*> <*> <*> RAS KERNEL INFO g...,5,0.006,0.173,0.526,28.81
1,2005-06-14 17:55:00,34,- <*> 2005.06.14 <*> <*> <*> RAS KERNEL FATAL ...,4,0.005,0.139,0.421,28.81
2,2005-07-25 23:30:00,88,- <*> 2005.07.25 <*> <*> <*> RAS KERNEL INFO g...,4,0.005,0.139,0.421,28.81
3,2005-06-14 17:55:00,33,- <*> 2005.06.14 <*> <*> <*> RAS KERNEL FATAL ...,3,0.004,0.104,0.316,28.81
4,2005-06-14 18:00:00,35,- <*> 2005.06.14 <*> <*> <*> RAS KERNEL FATAL ...,3,0.004,0.104,0.316,28.81
5,2005-06-14 18:30:00,41,- <*> 2005.06.14 <*> <*> <*> RAS KERNEL FATAL ...,3,0.004,0.104,0.316,28.81
6,2005-06-19 07:30:00,47,- <*> 2005.06.19 <*> <*> <*> RAS KERNEL INFO g...,3,0.004,0.104,0.316,28.81
7,2005-11-11 07:00:00,125,- <*> 2005.11.10 <*> <*> <*> RAS KERNEL INFO i...,3,0.004,0.104,0.316,28.81
8,2005-06-14 17:35:00,24,- <*> 2005.06.14 <*> <*> <*> RAS KERNEL FATAL ...,2,0.002,0.069,0.210,28.81
9,2005-07-11 21:00:00,75,KERNRTSP <*> 2005.07.11 <*> <*> <*> RAS KERNEL...,2,0.002,0.069,0.210,28.81


## 8. Template moi xuat hien khi nao?

Mot template moi duoc tinh tai lan dau tien template do xuat hien trong log stream. Day la signal quan trong vi no co the ung voi behavior moi, deploy moi, loi moi hoac pattern attack moi.

In [12]:
first_seen_df = (
    log_df
    .groupby("template_id", as_index=False)
    .agg(
        first_seen=("timestamp", "min"),
        last_seen=("timestamp", "max"),
        count=("template_id", "size"),
        labeled_anomaly_count=("is_labeled_anomaly", "sum"),
    )
    .sort_values("first_seen")
    .reset_index(drop=True)
)
first_seen_df["template"] = first_seen_df["template_id"].map(template_lookup)
first_seen_df["labeled_anomaly_count"] = first_seen_df["labeled_anomaly_count"].astype(int)
first_seen_df = first_seen_df[[
    "template_id", "template", "first_seen", "last_seen", "count", "labeled_anomaly_count"
]]

new_templates_path = RESULTS_DIR / "new_templates_first_seen.csv"
first_seen_df.to_csv(new_templates_path, index=False)

print(f"Tong so final template moi theo stream: {len(first_seen_df):,}")
print(f"Da export: {new_templates_path}")
first_seen_df.head(20)


Tong so final template moi theo stream: 151
Da export: results/new_templates_first_seen.csv


,template_id,template,first_seen,last_seen,count,labeled_anomaly_count
0,1,- <*> <*> <*> <*> <*> RAS KERNEL INFO instruct...,2005-06-03 22:42:50,2005-12-27 09:24:58,42,0
1,2,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> doub...,2005-06-03 23:47:20,2005-08-28 09:57:06,109,0
2,3,- <*> <*> <*> <*> <*> RAS KERNEL INFO CE sym <...,2005-06-04 01:21:59,2005-12-04 17:00:41,92,0
3,4,APPREAD <*> <*> <*> <*> <*> RAS APP FATAL ciod...,2005-06-04 07:24:32,2005-06-16 21:55:02,3,3
4,5,- <*> 2005.06.05 <*> <*> <*> RAS KERNEL INFO g...,2005-06-05 07:09:01,2005-06-05 17:29:13,44,0
5,6,- <*> <*> <*> <*> <*> RAS KERNEL FATAL force l...,2005-06-05 15:10:46,2005-06-14 17:59:48,2,0
6,7,- <*> 2005.06.06 <*> <*> <*> RAS KERNEL INFO g...,2005-06-07 02:50:03,2005-06-07 05:41:28,4,0
7,8,- 1118112700 2005.06.06 R23-M0-N0-I:J18-U01 20...,2005-06-07 02:51:40,2005-06-07 02:51:40,1,0
8,9,- <*> 2005.06.07 <*> <*> <*> RAS KERNEL INFO g...,2005-06-07 13:05:49,2005-06-08 05:18:17,15,0
9,10,- <*> <*> <*> <*> <*> RAS APP FATAL ciod: LOGI...,2005-06-08 01:15:58,2005-12-01 04:02:59,18,0


## 9. Optional: precision/recall neu dataset co label

Assignment ghi ro truong hop HDFS thi tinh precision/recall. Dataset dang dung la BGL, nhung BGL cung co label o token dau tien: `-` la normal, cac gia tri khac la anomaly category. Vi vay ta tinh precision/recall tham khao o muc log-line.

Cach map prediction:

- Mot window-template bi 3σ flag la spike.
- Moi log line nam trong dung window va template do duoc gan `predicted_anomaly=True`.

In [13]:
def compute_precision_recall(y_true, y_pred):
    y_true = pd.Series(y_true).astype(bool)
    y_pred = pd.Series(y_pred).astype(bool)

    tp = int((y_true & y_pred).sum())
    fp = int((~y_true & y_pred).sum())
    fn = int((y_true & ~y_pred).sum())
    tn = int((~y_true & ~y_pred).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


spike_keys = set()
if not spike_df.empty:
    spike_keys = set(zip(spike_df["timestamp"], spike_df["template_id"]))

eval_df = log_df.copy()
eval_df["window_start"] = eval_df["timestamp"].dt.floor(WINDOW)
eval_df["predicted_anomaly"] = [
    (window_start, template_id) in spike_keys
    for window_start, template_id in zip(eval_df["window_start"], eval_df["template_id"])
]

metrics = compute_precision_recall(eval_df["is_labeled_anomaly"], eval_df["predicted_anomaly"])
metrics_df = pd.DataFrame([metrics])
metrics_path = RESULTS_DIR / "log_anomaly_detection_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print(f"Da export: {metrics_path}")
metrics_df


Da export: results/log_anomaly_detection_metrics.csv


,tp,fp,fn,tn,precision,recall,f1
0,112,1604,31,253,0.065268,0.783217,0.120495


# Phase 3: Embedding + Cross-signal

Phase nay dung TF-IDF tren final templates de tinh similarity matrix va tao template clusters. Sau do inject mot dong log la vao Drain3 de kiem tra new-template detection.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

templates = template_df["template"].tolist()
template_ids = template_df["template_id"].tolist()

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b[\w.$:/-]+\b")
tfidf_matrix = vectorizer.fit_transform(templates)
similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_df = pd.DataFrame(similarity_matrix, index=template_ids, columns=template_ids)
similarity_path = RESULTS_DIR / "template_similarity_matrix.csv"
similarity_df.to_csv(similarity_path)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"Da export: {similarity_path}")
similarity_df.iloc[:5, :5]


TF-IDF matrix shape: (151, 641)
Similarity matrix shape: (151, 151)
Da export: results/template_similarity_matrix.csv


,73,85,2,3,77
73,1.000000,0.090273,0.098957,0.083030,0.294145
85,0.090273,1.000000,0.554126,0.052825,0.085372
2,0.098957,0.554126,1.000000,0.057907,0.093585
3,0.083030,0.052825,0.057907,1.000000,0.078522
77,0.294145,0.085372,0.093585,0.078522,1.000000


In [15]:
# Tim cac cap template giong nhau nhat, bo qua diagonal.
similar_pairs = []
for i in range(len(template_ids)):
    for j in range(i + 1, len(template_ids)):
        similar_pairs.append({
            "template_id_1": template_ids[i],
            "template_1": templates[i],
            "template_id_2": template_ids[j],
            "template_2": templates[j],
            "cosine_similarity": similarity_matrix[i, j],
        })

similar_pairs_df = (
    pd.DataFrame(similar_pairs)
    .sort_values("cosine_similarity", ascending=False)
    .reset_index(drop=True)
)
top_pairs_path = RESULTS_DIR / "top_template_similarities.csv"
similar_pairs_df.head(50).to_csv(top_pairs_path, index=False)

print(f"Da export: {top_pairs_path}")
similar_pairs_df.head(10)


Da export: results/top_template_similarities.csv


,template_id_1,template_1,template_id_2,template_2,cosine_similarity
0,55,- <*> 2005.06.26 <*> <*> <*> RAS KERNEL INFO p...,54,- <*> 2005.06.26 <*> <*> <*> RAS KERNEL INFO p...,1.000000
1,97,- <*> <*> <*> <*> <*> NULL DISCOVERY ERROR Nod...,94,- 1123030687 2005.08.02 R00-M0-N2 2005-08-02-1...,0.914905
2,102,APPSEV <*> <*> <*> <*> <*> RAS APP FATAL ciod:...,134,APPSEV <*> <*> <*> <*> <*> RAS APP FATAL ciod:...,0.907213
3,42,- <*> <*> <*> <*> <*> RAS KERNEL INFO total of...,142,- <*> <*> <*> <*> <*> RAS KERNEL INFO total of...,0.875810
4,11,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> ddr ...,139,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> ddr ...,0.752024
5,114,- <*> <*> <*> <*> <*> RAS APP FATAL ciod: Erro...,101,- <*> <*> <*> <*> <*> RAS APP FATAL ciod: Erro...,0.719618
6,126,- <*> <*> <*> <*> <*> RAS KERNEL INFO critical...,113,- <*> <*> <*> <*> <*> RAS KERNEL INFO critical...,0.705267
7,146,- <*> 2005.12.06 <*> <*> <*> RAS KERNEL INFO K...,141,- 1133638373 2005.12.03 R56-M0-NA-C:J15-U11 20...,0.701642
8,139,- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> ddr ...,142,- <*> <*> <*> <*> <*> RAS KERNEL INFO total of...,0.697388
9,49,- <*> <*> <*> <*> <*> RAS APP FATAL ciod: Erro...,114,- <*> <*> <*> <*> <*> RAS APP FATAL ciod: Erro...,0.687771


In [16]:
# Tao clusters bang connected components tren threshold cosine similarity.
# Threshold 0.35 giup gom nhung template co chung nhieu token domain nhu RAS/KERNEL/INFO/generating.
SIMILARITY_THRESHOLD = 0.35

parent = {tid: tid for tid in template_ids}

def find_parent(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    pa, pb = find_parent(a), find_parent(b)
    if pa != pb:
        parent[pb] = pa

for i in range(len(template_ids)):
    for j in range(i + 1, len(template_ids)):
        if similarity_matrix[i, j] >= SIMILARITY_THRESHOLD:
            union(template_ids[i], template_ids[j])

cluster_roots = {tid: find_parent(tid) for tid in template_ids}
root_to_cluster = {root: idx + 1 for idx, root in enumerate(sorted(set(cluster_roots.values())))}

cluster_df = template_df.copy()
cluster_df["tfidf_cluster_id"] = cluster_df["template_id"].map(lambda tid: root_to_cluster[cluster_roots[tid]])
cluster_summary_df = (
    cluster_df.groupby("tfidf_cluster_id")
    .agg(templates=("template_id", "count"), total_log_count=("count", "sum"))
    .sort_values(["templates", "total_log_count"], ascending=False)
    .reset_index()
)

clusters_path = RESULTS_DIR / "template_clusters_tfidf.csv"
cluster_df.to_csv(clusters_path, index=False)

print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"So TF-IDF clusters: {cluster_df['tfidf_cluster_id'].nunique()}")
print(f"Da export: {clusters_path}")
cluster_summary_df.head(10)


Similarity threshold: 0.35
So TF-IDF clusters: 62
Da export: results/template_clusters_tfidf.csv


,tfidf_cluster_id,templates,total_log_count
0,5,45,808
1,47,10,217
2,49,8,50
3,54,5,129
4,23,5,77
5,37,5,26
6,32,4,22
7,31,3,234
8,10,3,17
9,25,3,4


## Inject 1 dong log la va detect new template

Ta feed toan bo BGL log vao Drain3 truoc, sau do them mot dong log co structure khac han cac template cu. Neu `change_type` la `cluster_created`, Drain3 da tao new template.

In [17]:
injection_miner, _, _ = parse_logs(log_lines, sim_th=best_sim_th, depth=4)
before_clusters = len(injection_miner.drain.clusters)

weird_log = (
    "UNKNOWN_EVENT 9999999999 2099.01.01 R99-M9-N9-C:J99-U99 "
    "2099-01-01-00.00.00.000000 R99-M9-N9-C:J99-U99 RAS QUANTUM CRITICAL "
    "quantum cache resonance exceeded impossible_threshold=424242 phase=blue"
)

inject_result = injection_miner.add_log_message(weird_log)
after_clusters = len(injection_miner.drain.clusters)
is_new_template = after_clusters > before_clusters

inject_summary = pd.DataFrame([{
    "before_clusters": before_clusters,
    "after_clusters": after_clusters,
    "is_new_template": is_new_template,
    "change_type": inject_result["change_type"],
    "cluster_id": inject_result["cluster_id"],
    "template_mined": inject_result["template_mined"],
    "log_line": weird_log,
}])
inject_path = RESULTS_DIR / "injected_new_template_detection.csv"
inject_summary.to_csv(inject_path, index=False)

print(f"Clusters before injection: {before_clusters}")
print(f"Clusters after injection: {after_clusters}")
print(f"New template detected: {is_new_template}")
print(f"Da export: {inject_path}")
inject_summary


Clusters before injection: 151
Clusters after injection: 152
New template detected: True
Da export: results/injected_new_template_detection.csv


,before_clusters,after_clusters,is_new_template,change_type,cluster_id,template_mined,log_line
0,151,152,True,cluster_created,152,UNKNOWN_EVENT 9999999999 2099.01.01 R99-M9-N9-...,UNKNOWN_EVENT 9999999999 2099.01.01 R99-M9-N9-...


# Phase 4: Challenge - Build Mini Log Analyzer

Script can chay duoc:

```bash
python log_analyzer.py <logfile>
```

Output gom tong so dong, so template unique, top-5 template, spike trong 1 gio gan nhat, va new templates trong 1 gio gan nhat.

In [18]:
import subprocess
import sys

def run_log_analyzer(logfile):
    completed = subprocess.run(
        [sys.executable, "log_analyzer.py", str(logfile)],
        check=True,
        capture_output=True,
        text=True,
    )
    return completed.stdout

bgl_output = run_log_analyzer(DATA_PATH)
hdfs_path = Path("logpai loghub master HDFS/HDFS_2k.log")
hdfs_output = run_log_analyzer(hdfs_path)

print("=== BGL output ===")
print(bgl_output)
print("=== HDFS output ===")
print(hdfs_output)


=== BGL output ===
Log file: logpai loghub master BGL/BGL_2k.log
Total lines: 2,000
Unique templates: 151

Top-5 templates:
- [73] count=180 (9.00%): - <*> 2005.07.09 <*> <*> <*> RAS KERNEL INFO generating <*>
- [85] count=121 (6.05%): - <*> <*> <*> <*> <*> RAS KERNEL INFO <*> floating point alignment exceptions
- [2] count=109 (5.45%): - <*> <*> <*> <*> <*> RAS KERNEL INFO <*> double-hummer alignment exceptions
- [3] count=92 (4.60%): - <*> <*> <*> <*> <*> RAS KERNEL INFO CE sym <*> at <*> mask <*>
- [77] count=87 (4.35%): - <*> 2005.07.13 <*> <*> <*> RAS KERNEL INFO generating <*>

Templates spiking in the latest hour:
- None detected

New templates in the latest hour:
- None detected

=== HDFS output ===
Log file: logpai loghub master HDFS/HDFS_2k.log
Total lines: 2,000
Unique templates: 21

Top-5 templates:
- [2] count=314 (15.70%): <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
- [1] count=311 (15.55%): <*> <*> <

In [19]:
# So sanh output cua 2 dataset bang cung parser/config.
_, bgl_log_df, bgl_template_df = parse_logs(DATA_PATH.read_text(encoding="utf-8", errors="ignore").splitlines(), sim_th=0.5)
hdfs_lines = hdfs_path.read_text(encoding="utf-8", errors="ignore").splitlines()
_, hdfs_log_df, hdfs_template_df = parse_logs(hdfs_lines, sim_th=0.5)

comparison_df = pd.DataFrame([
    {
        "dataset": "BGL",
        "total_lines": len(bgl_log_df),
        "unique_templates": len(bgl_template_df),
        "top_template_share_pct": round(bgl_template_df.iloc[0]["count"] / len(bgl_log_df) * 100, 2),
        "reason": "Supercomputer log: nhieu component, severity, event type va hardware/kernel pattern nen template da dang hon.",
    },
    {
        "dataset": "HDFS",
        "total_lines": len(hdfs_log_df),
        "unique_templates": len(hdfs_template_df),
        "top_template_share_pct": round(hdfs_template_df.iloc[0]["count"] / len(hdfs_log_df) * 100, 2),
        "reason": "HDFS_2k lap lai nhieu thao tac block/DataNode/FSNamesystem nen it template hon.",
    },
])

comparison_path = RESULTS_DIR / "dataset_comparison_bgl_hdfs.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"Da export: {comparison_path}")
comparison_df


Da export: results/dataset_comparison_bgl_hdfs.csv


,dataset,total_lines,unique_templates,top_template_share_pct,reason
0,BGL,2000,151,9.0,"Supercomputer log: nhieu component, severity, ..."
1,HDFS,2000,21,15.7,HDFS_2k lap lai nhieu thao tac block/DataNode/...
